# Evaluation workflows

An **evaluation workflow** analyses a conversation after it has finished: it loads the call, works out a summary,
a success verdict or structured data, and stores what it finds. It never runs on a live call. A standard workflow
names one as its evaluator (`EvaluationConfig.evaluator_workflow_id`), and the server runs it once a conversation
ends.

Every workflow has a type, `WorkflowType`: `standard` (the workflows that run calls, and the default) or
`evaluation`.

This notebook:

1. Lists the templates and the ready-made analysis nodes
2. Shows that the analysis node types are offered only for evaluation workflows
3. Creates an evaluation workflow from a template, and finds it in the lists
4. Reads it back as an ordinary workflow
5. Names it as a standard workflow's evaluator, and sees that in `usage()`
6. Cleans up

Nothing here runs a workflow or calls an LLM.

> The typed config classes come from `interactly.configs` (requires `pip install "interactly[configs]"`).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient, WorkflowType

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

# Remove anything a previous, interrupted run of this notebook left behind.
for workflow_type in (WorkflowType.STANDARD, WorkflowType.EVALUATION):
    page = await client.workflows.list(search="22_evaluation_workflows", workflow_type=workflow_type, size=100)
    for leftover in await page.list_all():
        await client.workflows.delete(leftover.id)
        print("removed leftover:", leftover.name)

## 1. Templates and analysis nodes

`templates()` lists the starting points for a new evaluation workflow. `node_presets()` lists the ready-made
analysis nodes, in the order a workflow usually uses them; each holds a `config` ready to give a node.

In [ ]:
templates = await client.evaluation_workflows.templates()
for template in templates:
    print(f"{template.template_id:18s} {template.name}: {template.description}")

presets = await client.evaluation_workflows.node_presets()
print()
for preset in presets:
    print(f"{preset.node_type:18s} {preset.label}")

## 2. Analysis nodes are for evaluation workflows

`nodes.types(workflow_type=...)` lists the node types a workflow of that type can hold. The analysis nodes
(`conversation_fetch`, `analysis_llm`, `store_analysis`) are offered only for `evaluation`.

In [ ]:
ANALYSIS_NODES = {"conversation_fetch", "analysis_llm", "store_analysis"}


def type_names(node_types) -> set:
    return {t.get("type") if isinstance(t, dict) else getattr(t, "type", None) for t in node_types}


standard_types = type_names(await client.nodes.types())
evaluation_types = type_names(await client.nodes.types(workflow_type=WorkflowType.EVALUATION))
print("analysis nodes offered for a standard workflow   :", sorted(ANALYSIS_NODES & standard_types))
print("analysis nodes offered for an evaluation workflow:", sorted(ANALYSIS_NODES & evaluation_types))

## 3. Create one from a template

`create_from_template()` returns an ordinary `Workflow`, with version `0` active. It raises `NotFoundError` for an
unknown template, and `BadRequestError` when the template uses a node type this server has not enabled.

`evaluation_workflows.list()` lists the team's evaluation workflows with their versions (it is not paged).
`workflows.list()` lists standard workflows unless asked for `workflow_type="evaluation"`.

In [ ]:
from interactly.types.workflows.workflow import Workflow

template = next((t for t in templates if t.template_id == "default_framework"), templates[0])
evaluator: Workflow = await client.evaluation_workflows.create_from_template(
    template_id=template.template_id,
    name="Call review (22_evaluation_workflows)",
)
EVALUATOR_ID = evaluator.id
print(f"Created {evaluator.name!r} from {template.template_id!r}  id={EVALUATOR_ID}")
print(f"Active version: {evaluator.active_version_number}")

for item in await client.evaluation_workflows.list():
    if item.workflow_id == EVALUATOR_ID:
        print("\nevaluation_workflows.list():", item.name)
        for version in item.versions:
            print(f"  v{version.version_number} {version.version_name!r}  masks PHI: {version.has_phi_masking}")

evaluation_page = await client.workflows.list(search="22_evaluation_workflows", workflow_type=WorkflowType.EVALUATION)
standard_page = await client.workflows.list(search="22_evaluation_workflows")
print("\nin workflows.list(workflow_type='evaluation'):", any(w.id == EVALUATOR_ID for w in evaluation_page.items))
print("in the default workflows.list()              :", any(w.id == EVALUATOR_ID for w in standard_page.items))

## 4. It is an ordinary workflow

Once created, an evaluation workflow is managed through `client.workflows` like any other: versions, nodes,
edges, export. Its fully hydrated config shows the steps the template laid out.

In [ ]:
hydrated = await client.workflows.get_fully_hydrated(EVALUATOR_ID)
print("workflow_type:", hydrated.workflow_config.workflow_type)
for node in hydrated.node_configs:
    print(f"  {node.type:20s} {node.name}")

## 5. Name it as a workflow's evaluator

A standard workflow names its evaluator in `EvaluationConfig.evaluator_workflow_id`. `workflows.usage()` on the
evaluator then lists that workflow. A delete does not consult `usage()`, so check it before deleting an evaluator
other workflows rely on.

In [ ]:
from interactly.configs import (
    EvaluationConfig,
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)

greeter: Workflow = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(
            name="Greeter with a reviewer (22_evaluation_workflows)",
            evaluation_config=EvaluationConfig(evaluator_workflow_id=EVALUATOR_ID),
        ),
        node_configs=[
            SayStaticMessageNodeConfig(
                name="Greeting",
                is_start=True,
                static_messages_config=StaticMessagesConfig(static_messages=["Hello! Thanks for calling."]),
            )
        ],
        edge_configs=[],
    )
)
GREETER_ID = greeter.id

report = await client.workflows.usage(EVALUATOR_ID)
for entry in report.used_by:
    print(f"used by {entry.name!r} ({entry.workflow_id}), versions {entry.version_numbers}")

## 6. Cleanup

In [ ]:
await client.workflows.delete(GREETER_ID)
await client.workflows.delete(EVALUATOR_ID)
await client.close()
print("cleaned up")

## See also

- Guide: [`../docs/guides/workflows_and_versions.md`](../docs/guides/workflows_and_versions.md#evaluation-workflows) — evaluation workflows and `workflow_type`
- [`16_nodes_and_edges.ipynb`](16_nodes_and_edges.ipynb) — node types in depth
- [`03_workflow_versions.ipynb`](03_workflow_versions.ipynb) — versions, export and `usage()`